# Stage 8 — Final confirmation and one-time Nissan locked test

This is the last GPU experiment. It performs no architecture search.

1. Corrects the Stage 6 initialization confound for six essential ablations.
2. Trains frozen KAN and matched-MLP models with seeds 42, 52, 62, 72 and 82.
3. Writes a frozen-configuration artifact.
4. Only then opens the 26-trip Nissan locked test for final inference.
5. Seals the locked test and prohibits further tuning.

The locked split has 26 current-SOH labels, 8 valid 500-km loss labels, and no
valid 30-day loss labels. Therefore, the 30-day claim remains development-CV
evidence only.

Expected A100 runtime: approximately **25–60 minutes**.

In [ ]:
# BLOCK 1 — Configuration
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess, sys, time, zipfile

RUN_ID = "stage8_final_confirmation"
PRETRAIN_EPOCHS = 5
FINETUNE_EPOCHS = 30
SEEDS = [42, 52, 62, 72, 82]
INPUT_ROOT = Path("/content/dbpatchkan_stage8_input")
PROJECT_EXTRACT = Path("/content/dbpatchkan_stage8_project")
RUN_ROOT = Path("/content/dbpatchkan_stage8_runs") / RUN_ID

def update(message):
    stamp = datetime.now(timezone.utc).strftime("%H:%M:%S")
    print(f"[{stamp} UTC] {message}", flush=True)

update("BLOCK 1 starting")
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
update(
    f"BLOCK 1 complete: seeds={SEEDS}, "
    f"pretrain={PRETRAIN_EPOCHS}, finetune={FINETUNE_EPOCHS}"
)

In [ ]:
# BLOCK 2 — Upload the single Stage 8 package
from google.colab import files

update("BLOCK 2: upload DBPatchKAN_Stage8_Final_Confirmation_Upload.zip")
uploaded = files.upload()
zip_items = [
    (name, payload) for name, payload in uploaded.items()
    if name.lower().endswith(".zip")
]
assert len(zip_items) == 1, f"Upload exactly one ZIP; received {[x[0] for x in zip_items]}"
name, payload = zip_items[0]
package = INPUT_ROOT / "stage8_upload.zip"
package.write_bytes(payload)
update(f"Received {name} ({len(payload) / 2**20:.2f} MB)")

with zipfile.ZipFile(package) as archive:
    archive.extractall(INPUT_ROOT)
required = {
    "dbpatchkan_stage8_colab_code.zip",
    "development_101_trips.npz",
    "segments_test_UNTOUCHED.npz",
    "STAGE8_UPLOAD_MANIFEST.json",
}
located = {path.name: path for path in INPUT_ROOT.rglob("*") if path.is_file()}
missing = sorted(required - set(located))
assert not missing, f"Stage 8 package is incomplete: {missing}"
for filename in required:
    source = located[filename]
    destination = INPUT_ROOT / filename
    if source.resolve() != destination.resolve():
        shutil.copy2(source, destination)
update("BLOCK 2 complete: final package extracted")

In [ ]:
# BLOCK 3 — Install the project and verify A100
update("BLOCK 3 installing project")
if PROJECT_EXTRACT.exists():
    shutil.rmtree(PROJECT_EXTRACT)
PROJECT_EXTRACT.mkdir(parents=True)
with zipfile.ZipFile(INPUT_ROOT / "dbpatchkan_stage8_colab_code.zip") as archive:
    archive.extractall(PROJECT_EXTRACT)
roots = list(PROJECT_EXTRACT.rglob("pyproject.toml"))
assert len(roots) == 1, f"Expected one project root, found {roots}"
PROJECT_ROOT = roots[0].parent

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0"],
    check=True,
)
source_root = str(PROJECT_ROOT / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)
os.environ["PYTHONPATH"] = source_root + os.pathsep + os.environ.get("PYTHONPATH", "")

import torch, pandas as pd, dbpatchkan
assert torch.cuda.is_available(), "Enable Runtime > Change runtime type > A100 GPU"
update(
    f"BLOCK 3 complete: GPU={torch.cuda.get_device_name(0)}, "
    f"torch={torch.__version__}"
)

In [ ]:
# BLOCK 4 — Verify hashes without opening the locked arrays
import hashlib

update("BLOCK 4 validating immutable inputs")
manifest = json.loads((INPUT_ROOT / "STAGE8_UPLOAD_MANIFEST.json").read_text())
for filename, expected in manifest["files"].items():
    path = INPUT_ROOT / filename
    value = hashlib.sha256(path.read_bytes()).hexdigest()
    assert value == expected["sha256"], f"Hash mismatch: {filename}"

assert manifest["locked_test_included"] is True
assert manifest["locked_test_authorized_by_user"] is True
assert manifest["locked_test_label_counts"] == {
    "current_soh": 26,
    "loss_30d": 0,
    "loss_500km": 8,
}

DEVELOPMENT = INPUT_ROOT / "development_101_trips.npz"
LOCKED_TEST = INPUT_ROOT / "segments_test_UNTOUCHED.npz"

# Intentionally do not call np.load on LOCKED_TEST in this block.
update("BLOCK 4 complete: hashes valid; locked arrays remain unopened")

In [ ]:
# BLOCK 5 — Development-only corrected ablations and five-seed training
update("BLOCK 5 starting development-only final confirmation")
development_command = [
    sys.executable, "-u", "-m", "dbpatchkan.analysis.stage8_final",
    "--phase", "development",
    "--development-data", str(DEVELOPMENT),
    "--output-dir", str(RUN_ROOT),
    "--pretrain-epochs", str(PRETRAIN_EPOCHS),
    "--finetune-epochs", str(FINETUNE_EPOCHS),
]
update("COMMAND: " + " ".join(development_command))
started = time.time()
process = subprocess.Popen(
    development_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
assert return_code == 0, f"Development phase failed: {return_code}"
update(f"BLOCK 5 complete in {(time.time() - started) / 60:.1f} minutes")

In [ ]:
# BLOCK 6 — Review development evidence and verify model freeze
update("BLOCK 6 verifying frozen configuration")
ablation_metrics = pd.read_csv(
    RUN_ROOT / "corrected_ablations" / "corrected_ablation_metrics.csv"
)
ablation_bootstrap = pd.read_csv(
    RUN_ROOT / "corrected_ablations" / "corrected_ablation_bootstrap.csv"
)
seed_summary = pd.read_csv(
    RUN_ROOT / "final_models" / "final_multiseed_validation_summary.csv"
)
freeze = json.loads((RUN_ROOT / "FROZEN_CONFIGURATION.json").read_text())

assert freeze["selection_complete"] is True
assert freeze["locked_test_seen"] is False
assert freeze["registered_models"] == 10

display(
    ablation_metrics.groupby(["variant", "target"], as_index=False)
    .agg(mae_mean=("mae", "mean"), folds=("fold", "nunique"))
    .sort_values(["target", "mae_mean"])
)
display(seed_summary)
display(ablation_bootstrap)
print(json.dumps(freeze, indent=2))
update("BLOCK 6 complete: architecture frozen; locked test still unseen")

In [ ]:
# BLOCK 7 — Open and evaluate the locked Nissan test exactly once
update("BLOCK 7 opening locked test for final inference only")
locked_command = [
    sys.executable, "-u", "-m", "dbpatchkan.analysis.stage8_final",
    "--phase", "locked_test",
    "--development-data", str(DEVELOPMENT),
    "--locked-test-data", str(LOCKED_TEST),
    "--output-dir", str(RUN_ROOT),
]
started = time.time()
process = subprocess.Popen(
    locked_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
assert return_code == 0, f"Locked-test phase failed: {return_code}"

decision_command = [
    sys.executable, "-u", "-m", "dbpatchkan.analysis.stage8_final",
    "--phase", "decision",
    "--development-data", str(DEVELOPMENT),
    "--output-dir", str(RUN_ROOT),
]
subprocess.run(decision_command, check=True, env=os.environ.copy())
update(f"BLOCK 7 complete and sealed in {(time.time() - started) / 60:.1f} minutes")

In [ ]:
# BLOCK 8 — Display final locked-test results
update("BLOCK 8 loading final results")
locked_root = RUN_ROOT / "locked_test_once"
metrics = pd.read_csv(locked_root / "locked_test_metrics.csv")
seed_test_summary = pd.read_csv(
    locked_root / "locked_test_multiseed_summary.csv"
)
bootstrap = pd.read_csv(locked_root / "locked_test_paired_bootstrap.csv")
winners = pd.read_csv(RUN_ROOT / "FINAL_LOCKED_TEST_WINNERS.csv")
access = json.loads((locked_root / "LOCKED_TEST_COMPLETE.json").read_text())

display(metrics)
display(seed_test_summary)
display(bootstrap)
display(winners)
print(json.dumps(access, indent=2))
update("BLOCK 8 complete: these results are final and cannot be tuned")

In [ ]:
# BLOCK 9 — Consolidate prior evidence and create final inventory
update("BLOCK 9 consolidating thesis evidence")
prior_root = RUN_ROOT / "prior_evidence"
prior_root.mkdir(parents=True, exist_ok=True)
for path in INPUT_ROOT.glob("prior_*.csv"):
    shutil.copy2(path, prior_root / path.name)

inventory = []
for path in sorted(RUN_ROOT.rglob("*")):
    if path.is_file():
        inventory.append({
            "relative_path": path.relative_to(RUN_ROOT).as_posix(),
            "size_bytes": path.stat().st_size,
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        })
pd.DataFrame(inventory).to_csv(RUN_ROOT / "FINAL_ARTIFACT_INVENTORY.csv", index=False)

final_decision = json.loads((RUN_ROOT / "FINAL_STAGE8_DECISION.json").read_text())
assert final_decision["experiments_complete"] is True
assert final_decision["architecture_tuning_closed"] is True
print(json.dumps(final_decision, indent=2))
update(f"BLOCK 9 complete: {len(inventory)} artifacts indexed")

In [ ]:
# BLOCK 10 — ZIP and automatically download all final models and results
update("BLOCK 10 packaging final Stage 8 artifacts")
from google.colab import files

final_zip = Path("/content/DBPatchKAN_stage8_FINAL_CONFIRMATION_ALL_MODELS_RESULTS.zip")
if final_zip.exists():
    final_zip.unlink()
shutil.make_archive(
    str(final_zip.with_suffix("")),
    "zip",
    RUN_ROOT.parent,
    RUN_ROOT.name,
)
update(f"Final ZIP ready: {final_zip.stat().st_size / 2**20:.2f} MB")
files.download(str(final_zip))
update("BLOCK 10 complete: download requested; experiments are finished")